# 🌡️ Project 14 — Surface Temperature Map

## Goal

Create a map showing the **surface temperature** using the **Landsat 8 Collection 2 Level 2** thermal dataset.

Examples of hot and cool surfaces:

- 🌆 Cities → Hot
- 🌾 Cropland → Moderate
- 🌲 Forest → Cooler
- 💧 Lakes → Cool

---

# Why is this Useful in Remote Sensing?

Surface temperature is widely used in:

- 🌡 Urban Heat Island studies
- 🌾 Agricultural drought monitoring
- 🔥 Wildfire analysis
- 💧 Water resource management
- 🌍 Climate change research

For example:

```text
Land Surface Temperature
        +
Land Cover
        ↓
Urban Heat Island Analysis
```

Cities are often warmer than forests because **concrete and asphalt absorb more solar energy** than vegetation.

---

# New Skill: Working with Thermal Bands

Previously:

```text
Sentinel-2
      ↓
Visible & NIR Bands
      ↓
NDVI / NDWI
```

Today:

```text
Landsat 8
      ↓
Thermal Band
      ↓
Surface Temperature
      ↓
Temperature Map
```

Unlike NDVI or NDWI, we are **not calculating an index**.

Instead, we're using a **temperature band** that is already provided in the Landsat Level-2 product.

---

# Why Landsat Instead of Sentinel-2?

You have mainly used Sentinel-2.

Sentinel-2 provides bands such as:

- Blue
- Green
- Red
- NIR
- SWIR

These are excellent for vegetation, water, and land cover analysis.

However:

> **Sentinel-2 does not contain thermal infrared bands.**

Landsat 8 includes thermal sensors, making it suitable for estimating **land surface temperature**.

---

# Step 1 — Define the Study Area

We'll use Germany as an example.

```javascript
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);
```

Nothing new here.

The study area tells Earth Engine where to search for satellite imagery.

---

# Step 2 — Load the Landsat 8 Collection

```javascript
var image = ee.ImageCollection(
  'LANDSAT/LC08/C02/T1_L2'
)
.filterBounds(countryArea)
.filterDate('2024-07-01','2024-07-31')
.median()
.clip(countryArea);
```

This workflow should already look familiar.

```text
ImageCollection
      ↓
filterBounds()
      ↓
filterDate()
      ↓
median
      ↓
clip()
```

The only difference is the dataset.

Instead of Sentinel-2, we're using the **Landsat 8 Collection 2 Level 2** product.

---

# Step 3 — Select the Temperature Band ⭐

Today's new band is:

```javascript
var temperature = image.select('ST_B10');
```

Let's understand what **ST_B10** means.

```text
ST
```

↓

Surface Temperature

```text
B10
```

↓

Thermal Band 10

Unlike Sentinel-2 bands (B2, B3, B4, B8), this band stores **surface temperature information**.

---

# Step 4 — Convert to Degrees Celsius ⭐

The `ST_B10` values are **scaled integers**, not actual temperatures.

According to the Landsat documentation, convert them using:

```javascript
var temperatureC = temperature
  .multiply(0.00341802)
  .add(149.0)
  .subtract(273.15);
```

Let's understand the workflow.

```text
Raw Band Values
       ↓
Apply Scale Factor
       ↓
Kelvin
       ↓
Convert to Celsius
```

After this conversion:

```text
temperatureC
```

is an `ee.Image` where every pixel stores **land surface temperature in °C**.

Notice something important.

Just like NDVI:

```text
Image
      ↓
New Calculation
      ↓
New Image
```

The conversion creates a **new image**.

The original `temperature` image remains unchanged.

---

# Step 5 — Create a Visualization

```javascript
var tempVis = {
  min:10,
  max:40,
  palette:[
    'blue',
    'cyan',
    'green',
    'yellow',
    'orange',
    'red'
  ]
};
```

Interpretation:

- 🔵 Blue → Cool
- 🟦 Cyan → Slightly Cool
- 🟢 Green → Moderate
- 🟡 Yellow → Warm
- 🟠 Orange → Hot
- 🔴 Red → Very Hot

---

# Step 6 — Display the Temperature Map

```javascript
Map.centerObject(countryArea,5);

Map.addLayer(
  temperatureC,
  tempVis,
  'Surface Temperature (°C)'
);
```

Now zoom into different regions and compare:

- 🌲 Forests
- 💧 Lakes
- 🌆 Cities
- 🌾 Agricultural land

Notice how temperatures vary across the landscape.

---

# Complete Code

```javascript
// 1. Study Area
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);

// 2. Landsat 8 Image
var image = ee.ImageCollection(
  'LANDSAT/LC08/C02/T1_L2'
)
.filterBounds(countryArea)
.filterDate('2024-07-01','2024-07-31')
.median()
.clip(countryArea);

// 3. Select Temperature Band
var temperature = image.select('ST_B10');

// 4. Convert to Celsius
var temperatureC = temperature
  .multiply(0.00341802)
  .add(149.0)
  .subtract(273.15);

// 5. Visualization
var tempVis = {
  min:10,
  max:40,
  palette:[
    'blue',
    'cyan',
    'green',
    'yellow',
    'orange',
    'red'
  ]
};

// 6. Display
Map.centerObject(countryArea,5);

Map.addLayer(
  temperatureC,
  tempVis,
  'Surface Temperature (°C)'
);
```

---

# Understanding the Temperature Conversion

Think about each step:

```javascript
temperature
```

↓

Select the thermal band.

↓

```javascript
.multiply(0.00341802)
```

↓

Apply the scale factor.

↓

```javascript
.add(149.0)
```

↓

Convert the scaled values into **Kelvin**.

↓

```javascript
.subtract(273.15)
```

↓

Convert **Kelvin** to **Celsius**.

Result:

```text
Surface Temperature Image (°C)
```

Every pixel now stores an estimated land surface temperature.

---

# Visual Workflow

```text
Landsat 8 Image

↓

Thermal Band (ST_B10)

↓

Apply Scale Factor

↓

Kelvin

↓

Convert to Celsius

↓

Surface Temperature Map
```

---

# Things to Remember

## 1. Sentinel-2 vs Landsat 8

| Sentinel-2 | Landsat 8 |
|------------|-----------|
| Vegetation, water, land cover | Includes thermal data |
| No thermal band | Thermal Band (ST_B10) |
| NDVI, NDWI | Surface Temperature |

---

## 2. ST_B10

`ST_B10` is the **Surface Temperature** band provided in the Landsat 8 Level-2 product.

It **does not store temperature directly in °C**.

The values must be converted before interpretation.

---

## 3. Temperature Conversion

The workflow is:

```text
ST_B10
    ↓
Multiply
    ↓
Add Offset
    ↓
Kelvin
    ↓
Subtract 273.15
    ↓
Celsius
```

After conversion, every pixel represents the estimated **land surface temperature**.

---

## 4. Surface Temperature is an Image

Like NDVI and NDWI:

```text
One Pixel

↓

One Temperature Value
```

Every pixel has its own temperature.

It is **not** a single temperature for the whole study area.

---

## 5. Thermal Bands Measure Surface Temperature

Satellite thermal bands measure the temperature of the Earth's **surface**, such as:

- Soil
- Asphalt
- Water
- Vegetation
- Buildings

They do **not** measure the air temperature reported by weather stations.

---

# Common Beginner Mistakes

## ❌ Mistake 1: Using Sentinel-2 for Temperature

Sentinel-2 has **no thermal band**, so it cannot directly estimate land surface temperature.

Use Landsat 8 (or another satellite with thermal bands).

---

## ❌ Mistake 2: Displaying ST_B10 Directly

The raw `ST_B10` values are **scaled integers**, not temperatures.

Always convert them before visualization.

Correct:

```javascript
var temperatureC = temperature
  .multiply(0.00341802)
  .add(149.0)
  .subtract(273.15);
```

---

## ❌ Mistake 3: Confusing Air Temperature with Surface Temperature

Surface temperature and air temperature are **not the same**.

Satellites measure the temperature of the Earth's surface, not the air a few meters above the ground.

---

## ❌ Mistake 4: Choosing an Inappropriate Visualization Range

For most summer studies, a range such as:

```javascript
min:10,
max:40
```

works well.

Using a much wider range may reduce the contrast in your map.

---

# Your Practice (Project 14.1)

Create a surface temperature map for a country or city of your choice.

## Requirements

1. Create a study area.
2. Load the Landsat 8 Level-2 collection.
3. Filter by date and location.
4. Select the `ST_B10` band.
5. Convert it to degrees Celsius.
6. Display the temperature map.

---

Before writing the code, answer these two questions.

## Question 1

Why do we use **Landsat 8** instead of **Sentinel-2** to create a surface temperature map?

---

## Question 2

Why can't we display the `ST_B10` band directly without converting it first?

Send me your answers and your code, and I'll review both your understanding and your implementation exactly like a university professor.